### Setup and Configuration
Import libraries, locate the project root from any notebook working directory, and define the raw dataset/config paths.


In [17]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Find the repository root from the current notebook working directory."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "config.yaml").exists() and (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root. Run this notebook inside the project directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "CIC-ToN-IoT.csv"
SAMPLE_SIZE = 100_000

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH}")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset file not found: {DATA_PATH}")

print("Project root:", PROJECT_ROOT)
print("Config path:", CONFIG_PATH)
print("Dataset path:", DATA_PATH)


Project root: E:\Paper Multiclass-Intrusion-Detection-System
Config path: E:\Paper Multiclass-Intrusion-Detection-System\configs\config.yaml
Dataset path: E:\Paper Multiclass-Intrusion-Detection-System\data\raw\CIC-ToN-IoT.csv


### Load Configuration
Load expected dataset metadata, selected feature columns, drop columns, and label settings from `configs/config.yaml`.


In [18]:
with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

data_config = config["data"]
label_column = data_config["label_column"]
feature_columns = data_config["feature_columns"]
drop_columns = data_config["drop_columns"]

print("Expected rows:", data_config["expected_rows"])
print("Expected total columns:", data_config["expected_total_columns"])
print("Expected feature count:", data_config["expected_feature_count"])
print("Configured label column:", label_column)
print("Configured feature columns:", len(feature_columns))
print("Configured drop columns:", len(drop_columns))


Expected rows: 5351760
Expected total columns: 85
Expected feature count: 69
Configured label column: Attack
Configured feature columns: 69
Configured drop columns: 15


### Read Header and Sample
Read only the CSV header and a sample so this notebook remains fast and safe on a 3 GB dataset.


In [19]:
header_columns = pd.read_csv(DATA_PATH, nrows=0).columns.str.strip().tolist()
sample_df = pd.read_csv(DATA_PATH, nrows=SAMPLE_SIZE, low_memory=False)
sample_df.columns = sample_df.columns.str.strip()

print("Header columns:", len(header_columns))
print("Sample shape:", sample_df.shape)
display(sample_df.head())


Header columns: 85
Sample shape: (100000, 85)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Attack
0,177.30.87.144-192.168.1.1-0-0-0,177.30.87.144,0,192.168.1.1,0,0,25/04/2019 05:18:52 pm,47814343,5,0,...,1038036.0,0.000000e+00,1038036.0,1038036.0,5.187256e+14,8.984590e+14,1.556177e+15,1.657324e+07,0,Benign
1,167.49.176.28-50.165.192.168-0-0-0,167.49.176.28,0,50.165.192.168,0,0,25/04/2019 05:18:49 pm,2033142,2,0,...,0.0,0.000000e+00,0.0,0.0,1.556177e+15,0.000000e+00,1.556177e+15,1.556177e+15,0,Benign
2,230.158.52.59-177.21.192.168-0-0-0,230.158.52.59,0,177.21.192.168,0,0,25/04/2019 05:18:37 pm,82877133,14,0,...,1931160.5,1.711593e+06,3942470.0,226402.0,1.729085e+14,5.187256e+14,1.556177e+15,6.036493e+06,0,Benign
3,183.68.192.168-1.1.192.168-0-0-0,183.68.192.168,0,1.1.192.168,0,0,25/04/2019 05:18:42 pm,24359,2,0,...,0.0,0.000000e+00,0.0,0.0,1.556177e+15,0.000000e+00,1.556177e+15,1.556177e+15,0,Benign
4,183.41.192.168-1.1.192.168-0-0-0,183.41.192.168,0,1.1.192.168,0,0,25/04/2019 05:18:42 pm,10239351,3,0,...,4053975.0,0.000000e+00,4053975.0,4053975.0,7.780884e+14,1.100383e+15,1.556177e+15,6.185376e+06,0,Benign


### Validate Dataset Structure
Check that the raw dataset structure matches the proposal and the project configuration.


In [20]:
missing_config_features = sorted(set(feature_columns) - set(header_columns))
missing_drop_columns = sorted(set(drop_columns) - set(header_columns))
unknown_columns = sorted(set(header_columns) - set(feature_columns) - set(drop_columns) - {label_column})
feature_drop_overlap = sorted(set(feature_columns) & set(drop_columns))

print("Header column count matches config:", len(header_columns) == data_config["expected_total_columns"])
print("Feature count matches config:", len(feature_columns) == data_config["expected_feature_count"])
print("Missing configured features:", missing_config_features)
print("Missing configured drop columns:", missing_drop_columns)
print("Feature/drop overlap:", feature_drop_overlap)
print("Unmapped raw columns:", unknown_columns)

if label_column not in header_columns:
    raise ValueError(f"Label column '{label_column}' was not found in the raw dataset.")
if missing_config_features or missing_drop_columns or feature_drop_overlap or unknown_columns:
    raise ValueError("Dataset column mapping does not match config.yaml. Review the printed lists above.")


Header column count matches config: True
Feature count matches config: True
Missing configured features: []
Missing configured drop columns: []
Feature/drop overlap: []
Unmapped raw columns: []


### Inspect Sample Class Distribution
Display class counts in the sample as a quick sanity check before running the full chunked profile in notebook 01.


In [21]:
sample_class_distribution = (
    sample_df[label_column]
    .value_counts(dropna=False)
    .rename_axis("class")
    .reset_index(name="sample_count")
)
sample_class_distribution["sample_percentage"] = (
    sample_class_distribution["sample_count"] / sample_class_distribution["sample_count"].sum() * 100
)

display(sample_class_distribution)
print("Sample classes:", sample_class_distribution.shape[0])


,class,sample_count,sample_percentage
0,Benign,99126,99.126
1,mitm,489,0.489
2,ddos,202,0.202
3,dos,145,0.145
4,scanning,29,0.029
5,injection,7,0.007
6,password,2,0.002


Sample classes: 7


### Build Sample Variation Summary
Compute sample-level uniqueness, zero ratio, and standard deviation for the configured 67 model features.


In [22]:
X_sample = sample_df[feature_columns].apply(pd.to_numeric, errors="coerce")
X_sample = X_sample.replace([np.inf, -np.inf], np.nan)

variation_summary = pd.DataFrame({
    "feature": feature_columns,
    "sample_missing_count": X_sample.isna().sum().values,
    "sample_missing_percentage": (X_sample.isna().mean() * 100).values,
    "sample_unique_count": X_sample.nunique(dropna=False).values,
    "sample_zero_ratio": (X_sample == 0).mean().values,
    "sample_std": X_sample.std(numeric_only=True).values,
})

variation_summary = variation_summary.sort_values(
    by=["sample_unique_count", "sample_zero_ratio", "sample_std"],
    ascending=[True, False, True],
)

display(variation_summary.head(20))


,feature,sample_missing_count,sample_missing_percentage,sample_unique_count,sample_zero_ratio,sample_std
42,Fwd PSH Flags,0,0.0,2,0.99469,0.072676
64,Subflow Fwd Pkts,0,0.0,2,0.38706,0.487080
22,FIN Flag Cnt,0,0.0,3,0.20185,0.807076
58,Protocol,0,0.0,3,0.14891,3.305380
44,Fwd Seg Size Min,0,0.0,6,0.14891,7.629014
59,RST Flag Cnt,0,0.0,17,0.99598,35.603042
60,SYN Flag Cnt,0,0.0,55,0.97700,89.481468
54,Pkt Len Min,0,0.0,88,0.95219,35.657701
51,PSH Flag Cnt,0,0.0,89,0.96934,100.456381
20,Down/Up Ratio,0,0.0,90,0.71402,2167.509359


### Inspect Configured Drop Columns
Review the columns excluded from modeling and summarize their sample-level variation where numeric values are available.


In [23]:
drop_sample = sample_df[drop_columns].copy()
drop_summary_rows = []

for column in drop_columns:
    series = drop_sample[column]
    numeric_series = pd.to_numeric(series, errors="coerce")
    drop_summary_rows.append({
        "column": column,
        "sample_dtype": str(series.dtype),
        "sample_unique_count": series.nunique(dropna=False),
        "numeric_non_null_ratio": numeric_series.notna().mean(),
        "sample_zero_ratio": (numeric_series == 0).mean() if numeric_series.notna().any() else np.nan,
    })

drop_summary = pd.DataFrame(drop_summary_rows).sort_values(
    by=["numeric_non_null_ratio", "sample_unique_count"],
    ascending=[True, True],
)

display(drop_summary)


,column,sample_dtype,sample_unique_count,numeric_non_null_ratio,sample_zero_ratio
2,Dst IP,object,581,0.0,NaN
3,Timestamp,object,3075,0.0,NaN
1,Src IP,object,13791,0.0,NaN
0,Flow ID,object,20372,0.0,NaN
4,Bwd PSH Flags,int64,1,1.0,1.00000
5,Bwd URG Flags,int64,1,1.0,1.00000
8,Fwd Blk Rate Avg,int64,1,1.0,1.00000
9,Fwd Byts/b Avg,int64,1,1.0,1.00000
10,Fwd Pkts/b Avg,int64,1,1.0,1.00000
11,Fwd URG Flags,int64,1,1.0,1.00000


### Final Notebook Sanity Checks
Assert the key Phase 1 assumptions that should hold before moving into preprocessing.


In [24]:
assert len(header_columns) == data_config["expected_total_columns"]
assert len(feature_columns) == data_config["expected_feature_count"]
assert label_column in sample_df.columns
assert set(feature_columns).isdisjoint(drop_columns)
assert len(set(feature_columns) | set(drop_columns) | {label_column}) == len(header_columns)

print("Dataset variation notebook checks passed.")
print("Configured model features:", len(feature_columns))
print("Excluded non-model columns:", len(drop_columns))
print("Label column:", label_column)


Dataset variation notebook checks passed.
Configured model features: 69
Excluded non-model columns: 15
Label column: Attack
